In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path
import plotly.express as px

In [2]:
Data_DIR = Path("../Data/Raw")
REPORT_DIR = Path("../reports")
tickets = pd.read_csv(Data_DIR/"Customer_Service.csv")
tickets.head()


,Ticket_ID,Open_Date,Channel,Issue_Type,First_Response_Minutes,Resolution_Hours,Resolved,CSAT_1_to_5,Data_Class
0,TKT00001,2025-02-02,WhatsApp,Return request,6.5,9.3,Yes,2,Modeled customer service interaction
1,TKT00002,2024-01-01,Instagram,Order status,21.8,10.1,Yes,4,Modeled customer service interaction
2,TKT00003,2026-03-21,WhatsApp,Product quality,7.0,7.7,Yes,5,Modeled customer service interaction
3,TKT00004,2025-06-15,Phone,Product quality,4.7,26.3,Yes,4,Modeled customer service interaction
4,TKT00005,2026-07-05,Website,Exchange request,14.6,14.0,Yes,5,Modeled customer service interaction


In [3]:
tickets.shape

(6500, 9)

In [4]:
tickets.columns.tolist()

['Ticket_ID',
 'Open_Date',
 'Channel',
 'Issue_Type',
 'First_Response_Minutes',
 'Resolution_Hours',
 'Resolved',
 'CSAT_1_to_5',
 'Data_Class']

In [5]:
tickets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Ticket_ID               6500 non-null   object 
 1   Open_Date               6500 non-null   object 
 2   Channel                 6500 non-null   object 
 3   Issue_Type              6500 non-null   object 
 4   First_Response_Minutes  6500 non-null   float64
 5   Resolution_Hours        6500 non-null   float64
 6   Resolved                6500 non-null   object 
 7   CSAT_1_to_5             6500 non-null   int64  
 8   Data_Class              6500 non-null   object 
dtypes: float64(2), int64(1), object(6)
memory usage: 457.2+ KB


In [6]:
tickets.isna().sum()

Ticket_ID                 0
Open_Date                 0
Channel                   0
Issue_Type                0
First_Response_Minutes    0
Resolution_Hours          0
Resolved                  0
CSAT_1_to_5               0
Data_Class                0
dtype: int64

In [7]:
tickets["Resolved"].unique()

array(['Yes', 'No'], dtype=object)

In [8]:
tickets["Issue_Type"].unique()

array(['Return request', 'Order status', 'Product quality',
       'Exchange request', 'Size guidance', 'Payment',
       'Wholesale inquiry', 'General product question'], dtype=object)

In [9]:
tickets["Open_Date"] = pd.to_datetime(tickets["Open_Date"], errors="coerce")
tickets["Resolved"] = tickets["Resolved"].astype(str).str.strip().str.lower()
tickets["Low_CSAT"] = tickets["CSAT_1_to_5"] <= 3
tickets["Unresolved"] = tickets["Resolved"] == "no"
tickets["Return_Exchange"] = tickets["Issue_Type"].isin(["Return request","Exchange request"])
tickets["Wholesale_Inquiry"] = tickets["Issue_Type"] == "Wholesale inquiry"

In [10]:
ticket_quality = {}

ticket_quality["rows"] = tickets.shape[0]
ticket_quality["columns"] = tickets.shape[1]

ticket_quality["unique_ticket_ids"] = tickets["Ticket_ID"].nunique()
ticket_quality["duplicate_ticket_ids"] = int(tickets["Ticket_ID"].duplicated().sum())

ticket_quality["missing_open_date"] = int(tickets["Open_Date"].isna().sum())

ticket_quality["date_min"] = tickets["Open_Date"].min()
ticket_quality["date_max"] = tickets["Open_Date"].max()

ticket_quality["future_open_dates"] = int(
    (tickets["Open_Date"] > pd.Timestamp.today()).sum()
)

ticket_quality["csat_min"] = tickets["CSAT_1_to_5"].min()
ticket_quality["csat_max"] = tickets["CSAT_1_to_5"].max()
ticket_quality["csat_missing"] = int(tickets["CSAT_1_to_5"].isna().sum())

ticket_quality["negative_first_response"] = int(
    (tickets["First_Response_Minutes"] < 0).sum()
)

ticket_quality["negative_resolution_hours"] = int(
    (tickets["Resolution_Hours"] < 0).sum()
)

ticket_quality["first_response_above_60_minutes"] = int(
    (tickets["First_Response_Minutes"] > 60).sum()
)

ticket_quality["resolution_above_72_hours"] = int(
    (tickets["Resolution_Hours"] > 72).sum()
)

ticket_quality["unresolved_count"] = int(tickets["Unresolved"].sum())
ticket_quality["unresolved_rate"] = round(tickets["Unresolved"].mean() * 100, 2)

ticket_quality["low_csat_count"] = int(tickets["Low_CSAT"].sum())
ticket_quality["low_csat_rate"] = round(tickets["Low_CSAT"].mean() * 100, 2)

ticket_quality_df = pd.Series(ticket_quality).to_frame("Value")

ticket_quality_df
ticket_quality_df.to_csv("../reports/customer_service_quality_report.csv")

In [11]:
tickets["Channel"].value_counts(dropna=False)

Channel
Instagram    1666
Phone        1624
WhatsApp     1612
Website      1598
Name: count, dtype: int64

In [12]:
tickets["Issue_Type"].value_counts(dropna=False)

Issue_Type
Size guidance               1347
Order status                1273
General product question    1082
Exchange request             692
Product quality              668
Wholesale inquiry            599
Return request               526
Payment                      313
Name: count, dtype: int64

In [13]:
tickets["Resolved"].value_counts(dropna=False)

Resolved
yes    6180
no      320
Name: count, dtype: int64

In [14]:
tickets["CSAT_1_to_5"].value_counts(dropna=False).sort_index()

CSAT_1_to_5
1     337
2     435
3     929
4    2020
5    2779
Name: count, dtype: int64

In [15]:
issue_summary = tickets.groupby("Issue_Type").agg(
    Ticket_Count=("Ticket_ID", "count"),
    Avg_CSAT=("CSAT_1_to_5", "mean"),
    Low_CSAT_Rate=("Low_CSAT", "mean"),
    Unresolved_Rate=("Unresolved", "mean"),
    Avg_First_Response_Minutes=("First_Response_Minutes", "mean"),
    Avg_Resolution_Hours=("Resolution_Hours", "mean")
).round(2)

issue_summary = issue_summary.sort_values("Ticket_Count", ascending=False)

issue_summary
issue_summary.to_csv("../reports/issue_type_summary.csv")

In [16]:
customers = pd.read_csv(Data_DIR/"Customers.csv")
customers.head()

,Customer_ID,Customer_Type,Governorate,Age_Band,Gender,Join_Date,Acquisition_Channel,Consent_Status,Data_Class
0,C00001,Retail,Dakahlia,45-54,Male,2026-02-22,Organic Social,Not opted-in,Modeled/anonymized training customer - not act...
1,C00002,Retail,Beheira,35-44,Male,2024-09-30,Direct / WhatsApp,Opted-in,Modeled/anonymized training customer - not act...
2,C00003,Retail,Dakahlia,25-34,Female,2025-06-27,Organic Social,Opted-in,Modeled/anonymized training customer - not act...
3,C00004,Wholesale,Port Said,Business,Business,2024-09-27,Wholesale Sales,Opted-in,Modeled/anonymized training customer - not act...
4,C00005,Retail,Gharbia,55+,Female,2025-02-07,Paid Social,Not opted-in,Modeled/anonymized training customer - not act...


In [17]:
customers.shape

(30000, 9)

In [18]:
customers.describe()

,Customer_ID,Customer_Type,Governorate,Age_Band,Gender,Join_Date,Acquisition_Channel,Consent_Status,Data_Class
count,30000,30000,30000,30000,30000,30000,30000,30000,30000
unique,30000,2,13,6,4,1096,7,2,1
top,C00001,Retail,Alexandria,25-34,Female,2026-02-19,Direct / WhatsApp,Opted-in,Modeled/anonymized training customer - not act...
freq,1,28349,8415,10705,18931,45,6316,19187,30000


In [19]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Customer_ID          30000 non-null  object
 1   Customer_Type        30000 non-null  object
 2   Governorate          30000 non-null  object
 3   Age_Band             30000 non-null  object
 4   Gender               30000 non-null  object
 5   Join_Date            30000 non-null  object
 6   Acquisition_Channel  30000 non-null  object
 7   Consent_Status       30000 non-null  object
 8   Data_Class           30000 non-null  object
dtypes: object(9)
memory usage: 2.1+ MB


In [20]:
customers.columns.tolist()

['Customer_ID',
 'Customer_Type',
 'Governorate',
 'Age_Band',
 'Gender',
 'Join_Date',
 'Acquisition_Channel',
 'Consent_Status',
 'Data_Class']

In [21]:
customers["Join_Date"] = pd.to_datetime(customers["Join_Date"], errors="coerce")

In [22]:
customers.isna().sum()

Customer_ID            0
Customer_Type          0
Governorate            0
Age_Band               0
Gender                 0
Join_Date              0
Acquisition_Channel    0
Consent_Status         0
Data_Class             0
dtype: int64

In [23]:
customers.duplicated().sum()

np.int64(0)

In [24]:
customers["Customer_Type"].value_counts(dropna=False)

Customer_Type
Retail       28349
Wholesale     1651
Name: count, dtype: int64

In [25]:
customers["Governorate"].value_counts(dropna=False)

Governorate
Alexandria        8415
Cairo             5383
Giza              3307
Beheira           2374
Dakahlia          1851
Other             1807
Gharbia           1778
Sharqia           1509
Kafr El Sheikh    1197
Monufia            923
Damietta           599
Port Said          575
Suez               282
Name: count, dtype: int64

In [26]:
customers["Acquisition_Channel"].value_counts(dropna=False)

Acquisition_Channel
Direct / WhatsApp    6316
Organic Social       5639
Store Walk-in        5242
Paid Social          5175
Google Search        3401
Referral             2576
Wholesale Sales      1651
Name: count, dtype: int64

In [27]:
customer_quality = {}

customer_quality["rows"] = customers.shape[0]
customer_quality["columns"] = customers.shape[1]

customer_quality["unique_customer_ids"] = customers["Customer_ID"].nunique()
customer_quality["duplicate_customer_ids"] = int(customers["Customer_ID"].duplicated().sum())

customer_quality["missing_join_date"] = int(customers["Join_Date"].isna().sum())

customer_quality["join_date_min"] = customers["Join_Date"].min()
customer_quality["join_date_max"] = customers["Join_Date"].max()

customer_quality["future_join_dates"] = int(
    (customers["Join_Date"] > pd.Timestamp.today()).sum()
)

customer_quality["retail_count"] = int((customers["Customer_Type"] == "Retail").sum())
customer_quality["wholesale_count"] = int((customers["Customer_Type"] == "Wholesale").sum())

customer_quality_df = pd.Series(customer_quality).to_frame("Value")

customer_quality_df

,Value
rows,30000
columns,9
unique_customer_ids,30000
duplicate_customer_ids,0
missing_join_date,0
join_date_min,2023-10-01 00:00:00
join_date_max,2026-09-30 00:00:00
future_join_dates,0
retail_count,28349
wholesale_count,1651


In [28]:
customer_quality = {}

customer_quality["rows"] = customers.shape[0]
customer_quality["columns"] = customers.shape[1]

customer_quality["unique_customer_ids"] = customers["Customer_ID"].nunique()
customer_quality["duplicate_customer_ids"] = int(customers["Customer_ID"].duplicated().sum())

customer_quality["missing_join_date"] = int(customers["Join_Date"].isna().sum())

customer_quality["join_date_min"] = customers["Join_Date"].min()
customer_quality["join_date_max"] = customers["Join_Date"].max()

customer_quality["future_join_dates"] = int(
    (customers["Join_Date"] > pd.Timestamp.today()).sum()
)

customer_quality["retail_count"] = int((customers["Customer_Type"] == "Retail").sum())
customer_quality["wholesale_count"] = int((customers["Customer_Type"] == "Wholesale").sum())

customer_quality_df = pd.Series(customer_quality).to_frame("Value")

customer_quality_df
customer_quality_df.to_csv("../reports/customers_quality_report.csv")

Check Wholesale Customers:  because wholesale customers may represent revenue opportunities.

In [29]:
wholesale_customers = customers[customers["Customer_Type"] == "Wholesale"]

wholesale_customers.head()

,Customer_ID,Customer_Type,Governorate,Age_Band,Gender,Join_Date,Acquisition_Channel,Consent_Status,Data_Class
3,C00004,Wholesale,Port Said,Business,Business,2024-09-27,Wholesale Sales,Opted-in,Modeled/anonymized training customer - not act...
15,C00016,Wholesale,Alexandria,Business,Business,2024-11-12,Wholesale Sales,Not opted-in,Modeled/anonymized training customer - not act...
36,C00037,Wholesale,Alexandria,Business,Business,2026-03-10,Wholesale Sales,Opted-in,Modeled/anonymized training customer - not act...
40,C00041,Wholesale,Alexandria,Business,Business,2024-10-01,Wholesale Sales,Not opted-in,Modeled/anonymized training customer - not act...
48,C00049,Wholesale,Monufia,Business,Business,2025-09-10,Wholesale Sales,Opted-in,Modeled/anonymized training customer - not act...


In [30]:
wholesale_customers["Governorate"].value_counts(dropna=False)

Governorate
Alexandria        484
Cairo             295
Giza              194
Beheira           132
Dakahlia           96
Gharbia            94
Other              89
Sharqia            67
Kafr El Sheikh     64
Monufia            43
Port Said          35
Damietta           34
Suez               24
Name: count, dtype: int64

In [31]:
wholesale_customers["Acquisition_Channel"].value_counts(dropna=False)

Acquisition_Channel
Wholesale Sales    1651
Name: count, dtype: int64

In [32]:
wholesale_customers["Consent_Status"].value_counts(dropna=False)

Consent_Status
Opted-in        1075
Not opted-in     576
Name: count, dtype: int64

Chart 1: Tickets by Issue Type


In [ ]:
issue_counts= tickets["Issue_Type"].value_counts().reset_index()
issue_counts.columns = ["Issue_Type", "Ticket_Count"]

fig = px.bar(issue_counts, x="Ticket_Count", y="Issue_Type", title="Ticket Volume by Issue Type")
fig.show()

In [58]:
channel_counts = tickets["Channel"].value_counts().reset_index()

fig = px.bar(
    channel_counts,
    x="Channel",
    y="count",
    title="Ticket Volume by Channel",
    text_auto=True,          # Show exact numbers on bars
    range_y=[1500, 1700],    # Zoom in on the 1500-1700 range
    
)

fig.show()

In [38]:
tickets.columns.tolist()

['Ticket_ID',
 'Open_Date',
 'Channel',
 'Issue_Type',
 'First_Response_Minutes',
 'Resolution_Hours',
 'Resolved',
 'CSAT_1_to_5',
 'Data_Class',
 'Low_CSAT',
 'Unresolved',
 'Return_Exchange',
 'Wholesale_Inquiry']

Chart 3: CSAT Distribution

In [63]:
csat_counts = tickets["CSAT_1_to_5"].value_counts().sort_index().reset_index()
csat_counts
fig = px.bar(csat_counts, x = "CSAT_1_to_5", y="count" ,text_auto=True)
fig.show()

Chart 4: Customers by Customer Type


In [68]:
customer_type_counts = customers["Customer_Type"].value_counts().reset_index()
customer_type_counts.columns = ["Customer_Type", "Customer_Count"]

fig = px.pie(
    customer_type_counts,
    names="Customer_Type",
    values="Customer_Count",
    title="Customers by Customer Type"

)

fig.show()

Chart 5: Customers by Governorate


In [69]:
governorate_counts = customers["Governorate"].value_counts().reset_index()
governorate_counts.columns = ["Governorate", "Customer_Count"]

fig = px.bar(
    governorate_counts,
    x="Customer_Count",
    y="Governorate",
    orientation="h",
    title="Customers by Governorate"
)

fig.update_layout(height=600, yaxis={"categoryorder": "total ascending"})

fig.show()